In [ ]:
import json
import os.path as op
from collections.abc import Callable, Iterable
from glob import glob
from typing import Any, Iterable, Mapping

import numpy as np
import pandas as pd

In [ ]:
BASE_DIR = "" # Write your path here


def build_qc_table(
    files: str | Iterable[str],
    template: Mapping[str, str],
    key_fn: Callable[[dict], str | None],
    allowed_tasks: set[str] | None = None,
    per_file_extras: Callable[[str], Mapping[str, Any]] | None = None,
    record_extras: Mapping[str, Callable[[dict], Any]] | None = None,
    bad_types: tuple[str, ...] = ("skull_strip_report", "t1_norm_rpt"),
    ignore_uncertain: bool = True,
    uncertain_types: tuple[str, ...] = ("tsnr_rpt", "ica_aroma"),
) -> pd.DataFrame:
    """
    Build a QC table from one or many exclude.json files.

    Parameters
    ----------
    files
        Glob pattern or iterable of file paths to JSON reports.
    template
        Dict of QC keys -> default "none"/"good"/"bad".
    key_fn
        Function mapping a JSON record to the QC key to update, or None to skip.
    allowed_tasks
        If provided, only records whose `task` is in this set are considered (except bad_types).
    per_file_extras
        Function mapping a file path to extra columns to add for each record in that file.
    record_extras
        Map of column -> function extracting value from each record.
    bad_types
        JSON `type`s that, when rated "bad", mark all QC keys as "bad".
    ignore_uncertain
        When True, records with rating "uncertain" and type in `uncertain_types`
        do not update QC keys (extras are still recorded).
    uncertain_types
        JSON `type`s for which "uncertain" ratings are ignored when `ignore_uncertain` is True.

    Returns
    -------
    pandas.DataFrame
        QC table with one row per subject and a `subject` column.
    """
    file_list = glob(files) if isinstance(files, str) else list(files)
    qc_pd = pd.DataFrame()
    for path in file_list:
        with open(path, "r") as fh:
            qc_json = json.load(fh)
        pfv = per_file_extras(path) if per_file_extras else {}
        qc_dict: dict[str, dict[str, str]] = {}
        for rec in qc_json:
            subj = rec["sub"]
            qc_tmp = qc_dict.get(subj, dict(template))

            rating = rec.get("rating")
            rtype = rec.get("type")

            if rtype in bad_types and rating == "bad":
                for k in qc_tmp:
                    qc_tmp[k] = "bad"
            else:
                task = rec.get("task")
                if allowed_tasks is None or task in allowed_tasks:
                    key = key_fn(rec)
                    skip_uncertain = (
                        ignore_uncertain
                        and rating == "uncertain"
                        and rtype in uncertain_types
                    )
                    if key and not skip_uncertain:
                        cur = qc_tmp.get(key)
                        if cur in ("good", "none") and rating and rating != "none":
                            qc_tmp[key] = rating

            if record_extras:
                for k, fn in record_extras.items():
                    val = fn(rec)
                    # only set if val is not None/NaN and existing isn't already a non-empty value
                    if val is not None and not (
                        isinstance(val, float) and np.isnan(val)
                    ):
                        cur = qc_tmp.get(k)
                        if (
                            cur is None
                            or cur == ""
                            or (isinstance(cur, float) and np.isnan(cur))
                        ):
                            qc_tmp[k] = val
            for k, v in pfv.items():
                qc_tmp[k] = v
            qc_dict[subj] = qc_tmp
        qc_pd = pd.concat([qc_pd, pd.DataFrame(qc_dict).T])
    qc_pd["subject"] = qc_pd.index
    return qc_pd


def to_long_qc(
    df: pd.DataFrame,
    qc_keys: Iterable[str],
    drop_none: bool = False,
    keep_cols: tuple[str, ...] = ("fu",),
) -> pd.DataFrame:
    """
    Convert wide QC to long with columns: subject, rating, task, optional ses and keep_cols.

    Parameters
    ----------
    df
        Wide QC dataframe containing template keys and optional extra columns.
    qc_keys
        Columns that are QC keys (e.g., task-run or task-ses).
    drop_none
        If True, drop rows with rating == "none".
    keep_cols
        Extra columns to preserve in the long output if present (default keeps 'fu').

    Returns
    -------
    pandas.DataFrame
        Long QC dataframe with columns: subject, rating, task, [ses], [keep_cols...].
    """
    keep_existing = [c for c in keep_cols if c in df.columns]
    m = df.loc[:, ["subject", *keep_existing, *qc_keys]].melt(
        id_vars=["subject", *keep_existing],
        var_name="key",
        value_name="rating",
    )
    if drop_none:
        m = m[m["rating"] != "none"]
    parts = m["key"].str.split("-", n=1, expand=True)
    m["task"] = parts[0]
    if parts.shape[1] == 2:
        m["ses"] = parts[1]
    cols = ["subject", "rating", "task"]
    if "ses" in m.columns and m["ses"].notna().any():
        cols.append("ses")
    cols += keep_existing
    return m[cols].reset_index(drop=True)


def save_qc_long(
    *,
    files,
    template,
    key_fn,
    out_csv,
    allowed_tasks=None,
    per_file_extras=None,
    record_extras=None,
    bad_types=("skull_strip_report", "t1_norm_rpt"),
    ignore_uncertain=True,
    uncertain_types=("tsnr_rpt", "ica_aroma"),
    drop_none=False,
    keep_cols=("fu",),
) -> None:
    """
    Build wide QC then save long QC with subject, rating, task, optional ses and keep_cols.
    """
    wide = build_qc_table(
        files=files,
        template=template,
        key_fn=key_fn,
        allowed_tasks=allowed_tasks,
        per_file_extras=per_file_extras,
        record_extras=record_extras,
        bad_types=bad_types,
        ignore_uncertain=ignore_uncertain,
        uncertain_types=uncertain_types,
    )
    long_df = to_long_qc(
        wide, template.keys(), drop_none=drop_none, keep_cols=keep_cols
    )
    long_df.to_csv(out_csv, index=False)

In [ ]:
# ABCD-2023
qc_template = {
    "nback-01": "none",
    "nback-02": "none",
    "mid-01": "none",
    "mid-02": "none",
}

save_qc_long(
    files=op.join(BASE_DIR, "qc-reports/exclude-abcd-2023/*_exclude.json"),
    template={
        "nback-01": "none",
        "nback-02": "none",
        "mid-01": "none",
        "mid-02": "none",
    },
    key_fn=lambda r: f"{r['task']}-{int(r['run']):02d}" if "run" in r else None,
    per_file_extras=lambda _: {"fu": "2YearFollowUpYArm1"},
    out_csv=op.join(BASE_DIR, "dsets/abcd-2023_qc.csv"),
)

In [ ]:
# ABCD-2025
qc_template = {
    "nback-01": "none",
    "nback-02": "none",
    "mid-01": "none",
    "mid-02": "none",
}
save_qc_long(
    files=op.join(BASE_DIR, "qc-reports/exclude-abcd-2025/exclude_chunk-*.json"),
    template=qc_template,
    key_fn=lambda r: f"{r['task']}-{int(r['run']):02d}" if "run" in r else None,
    record_extras={"fu": lambda r: r.get("ses")},
    out_csv=op.join(BASE_DIR, "dsets/abcd-2025_qc.csv"),
)

In [ ]:
# IntegraMooDS
qc_template = {"nback": "none", "reward": "none"}
save_qc_long(
    files=op.join(BASE_DIR, "qc-reports/exclude-integramoods/exclude_*.json"),
    template=qc_template,
    key_fn=lambda r: r["task"] if r.get("task") in {"nback", "reward"} else None,
    allowed_tasks={"nback", "reward"},
    out_csv=op.join(BASE_DIR, "dsets/integramoods_qc.csv"),
)

In [ ]:
# QTIM
qc_template = {"nback-01": "none", "nback-02": "none"}
save_qc_long(
    files=op.join(BASE_DIR, "qc-reports/exclude-qtim/qtim_exclusion_chunk*.json"),
    template=qc_template,
    key_fn=lambda r: f"{r['task']}-{r['ses']}"
    if r.get("task") == "nback" and "ses" in r
    else None,
    allowed_tasks={"nback"},
    out_csv=op.join(BASE_DIR, "dsets/qtim_qc.csv"),
)

In [ ]:
# PIOP1 & PIOP2
qc_template = {"workingmemory": "none"}
for dname in ["piop1", "piop2"]:
    for fpath in glob(
        op.join(BASE_DIR, f"qc-reports/exclude-{dname}/exclude-{dname}.json")
    ):
        save_qc_long(
            files=[fpath],
            template=qc_template,
            key_fn=lambda r: "workingmemory"
            if r.get("task") == "workingmemory"
            else None,
            allowed_tasks={"workingmemory"},
            out_csv=op.join(BASE_DIR, f"dsets/{dname}_qc.csv"),
        )

In [ ]:
# HCP-YA
qc_template = {
    "GAMBLING-RL": "none",
    "GAMBLING-LR": "none",
    "WM-LR": "none",
    "WM-RL": "none",
}
save_qc_long(
    files=op.join(BASE_DIR, "qc-reports/exclude-hcpya/hcp_exclude_*.json"),
    template=qc_template,
    key_fn=lambda r: f"{r['task']}-{r['ses']}"
    if r.get("task") in {"GAMBLING", "WM"} and "ses" in r
    else None,
    allowed_tasks={"GAMBLING", "WM"},
    out_csv=op.join(BASE_DIR, "dsets/hcp-ya_qc.csv"),
)

In [ ]:
# HCP-D
qc_template = {"GUESSING-AP": "none", "GUESSING-PA": "none"}
save_qc_long(
    files=op.join(BASE_DIR, "qc-reports/exclude-hcpd/ds-hcpd_chunk-*_exclude.json"),
    template=qc_template,
    key_fn=lambda r: f"{r['task']}-{r['ses']}"
    if r.get("task") == "GUESSING" and "ses" in r
    else None,
    allowed_tasks={"GUESSING"},
    out_csv=op.join(BASE_DIR, "dsets/hcpd_qc.csv"),
)

In [ ]:
# DynaMORE
qc_template = {"mid": "none"}
save_qc_long(
    files=op.join(BASE_DIR, "qc-reports/exclude-dynamore/exclude-dynamore.json"),
    template=qc_template,
    key_fn=lambda r: "mid" if r.get("task") == "mid" else None,
    allowed_tasks={"mid"},
    out_csv=op.join(BASE_DIR, "dsets/dynamore_qc.csv"),
)

In [ ]:
# PNC
qc_template = {"frac2back": "none"}
save_qc_long(
    files=op.join(BASE_DIR, "qc-reports/exclude-pnc/exclude_chunk-*_emin_lea.json"),
    template=qc_template,
    key_fn=lambda r: "frac2back" if r.get("task") == "frac2back" else None,
    allowed_tasks={"frac2back"},
    out_csv=op.join(BASE_DIR, "dsets/pnc_qc.csv"),
)

In [ ]:
# WAHN
qc_template = {"nback": "none"}
save_qc_long(
    files=op.join(BASE_DIR, "qc-reports/exclude-wahn/exclude-wahn.json"),
    template=qc_template,
    key_fn=lambda r: "nback" if r.get("task") == "nback" else None,
    allowed_tasks={"nback"},
    out_csv=op.join(BASE_DIR, "dsets/wahn_qc.csv"),
)

In [ ]:
# IMAGEN
# BL, FU2 and FU3
qc_template = {"mid": "none"}
for fu in ["bl", "fu2", "fu3"]:
    save_qc_long(
        files=op.join(BASE_DIR, f"qc-reports/exclude-imagen/{fu}/*.json"),
        template=qc_template,
        key_fn=lambda r: "mid" if r.get("task") == "mid" else None,
        allowed_tasks={"mid"},
        per_file_extras=lambda p: {"site": op.basename(p).split("_")[2]},
        out_csv=op.join(BASE_DIR, f"dsets/imagen-{fu}_qc.csv"),
    )

In [ ]:
# ds003849
qc_template = {"nback": "none"}
save_qc_long(
    files=op.join(BASE_DIR, "qc-reports/exclude-ds003849/exclude-ds003849.json"),
    template=qc_template,
    key_fn=lambda r: "nback" if r.get("task") == "nback" else None,
    allowed_tasks={"nback"},
    out_csv=op.join(BASE_DIR, "dsets/ds003849_qc.csv"),
)

In [ ]:
# ds003858
qc_template = {"MID": "none"}
save_qc_long(
    files=op.join(BASE_DIR, "qc-reports/exclude-ds003858/exclude-ds003858.json"),
    template=qc_template,
    key_fn=lambda r: "MID" if r.get("task") == "MID" else None,
    allowed_tasks={"MID"},
    out_csv=op.join(BASE_DIR, "dsets/ds003858_qc.csv"),
)

In [ ]:
# ds005479
qc_template = {"MID": "none"}
save_qc_long(
    files=op.join(BASE_DIR, "qc-reports/exclude-ds005479/exclude-ds005479.json"),
    template=qc_template,
    key_fn=lambda r: "MID" if r.get("task") == "MID" else None,
    allowed_tasks={"MID"},
    out_csv=op.join(BASE_DIR, "dsets/ds005479_qc.csv"),
)

In [ ]:
# CHCP
# Mean RMS of movement is less than 0.5
qc_pd = pd.read_csv(op.join(BASE_DIR, "qc-reports/exclude-chcp/motionvals-chcp.csv"))
qc_pd["rating"] = np.where(qc_pd["fd_mean"] < 0.5, "good", "bad")
qc_pd["task"] = qc_pd["task"].replace({"Nback": "wm", "Gambling": "reward"})
qc_pd.to_csv(op.join(BASE_DIR, "dsets/chcp_qc.csv"), index=False)
